# Solution: Sigmoid Function & Logistic Regression (Extended Lab)

**Based on:** Coursera / DeepLearning.AI C1_W3_Lab02 (Optional Lab: Logistic Regression)  
**Companion to:** `Sigmoid_Logistic_Regression_Practice_Skeleton.ipynb`

This notebook contains complete implementations, alternate solutions, extra practice answers, simulation results, and discussion notes.


## 📋 Cheat Sheet

### Core Formulas
| Name | Formula | Notes |
|------|---------|-------|
| Sigmoid / Logistic | \( g(z) = \frac{1}{1+e^{-z}} \) | Maps ℝ → (0,1) |
| Logistic model | \( f_{\mathbf{w},b}(\mathbf{x}) = g(\mathbf{w}\cdot\mathbf{x}+b) \) | Probability \( P(y=1\mid x) \) |
| Decision | \( \hat{y} = 1 \) if \( f \ge \tau \) else 0 | Default \( \tau = 0.5 \) |
| Log-odds / logit | \( \log\frac{p}{1-p} = \mathbf{w}\cdot\mathbf{x}+b \) | Linear in features |

### NumPy essentials
```python
import numpy as np
np.exp(z)
1 / (1 + np.exp(-z))
np.clip(z, -500, 500)  # numerical stability
```

### Quick checks
- \( g(0) = 0.5 \)
- Asymptotes at 0 and 1
- \( g'(z) = g(z)(1-g(z)) \)

![Flowchart](logistic_sigmoid_flowchart.png)


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit   # alternate stable sigmoid

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
np.set_printoptions(precision=3, suppress=True)
print("Setup complete.")


## 1. Sigmoid Implementation (Solution)


In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid of z.
    Works for scalar or ndarray.
    """
    g = 1 / (1 + np.exp(-z))
    return g

# Test
z_test = np.array([-10, -1, 0, 1, 10])
print("z      :", z_test)
print("sigmoid:", sigmoid(z_test))
print("Expected roughly: [0.    0.269 0.5   0.731 1.   ]")


## 2. Visualize the Sigmoid (Solution)

In [ ]:
z_tmp = np.linspace(-10, 10, 200)
y = sigmoid(z_tmp)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(z_tmp, y, 'b-', lw=2.5, label='sigmoid(z)')
ax.axhline(0.5, color='orange', ls='--', lw=1.5, label='threshold = 0.5')
ax.axvline(0, color='gray', ls=':', lw=1)
ax.set_xlabel('z = w·x + b')
ax.set_ylabel('g(z) = P(y=1 | x)')
ax.set_title('Sigmoid / Logistic Function')
ax.legend(loc='upper left')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Logistic Regression Probability (Solution)


In [ ]:
def predict_proba(x, w, b):
    """
    Return probability predictions for logistic regression.
    x: (m,) or (m,1); w, b scalars for 1-D case.
    """
    z = w * x + b
    p = sigmoid(z)
    return p

# Demo
x_demo = np.array([0., 1, 2, 3, 4, 5])
print(predict_proba(x_demo, w=1.8, b=-5.4))


## 4. Tumor Classification Example (Solution)


In [ ]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

w_in = 1.8
b_in = -5.4

probs = predict_proba(x_train, w_in, b_in)
preds = (probs >= 0.5).astype(int)

print("x     :", x_train)
print("y     :", y_train)
print("P(y=1):", np.round(probs, 3))
print("pred  :", preds)
print("Accuracy:", (preds == y_train).mean())

# Plot
x_line = np.linspace(-0.5, 6, 100)
p_line = predict_proba(x_line, w_in, b_in)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_train[y_train==0], y_train[y_train==0], c='C0', s=90, label='y=0 (benign)', zorder=5)
ax.scatter(x_train[y_train==1], y_train[y_train==1], c='C3', s=90, label='y=1 (malignant)', zorder=5)
ax.plot(x_line, p_line, 'g-', lw=2.5, label=r'$f(x)=g(wx+b)$')
ax.axhline(0.5, color='orange', ls='--', label='threshold 0.5')
ax.set_xlabel('Tumor size (x)')
ax.set_ylabel('Probability / Label')
ax.set_title('Logistic Regression — Tumor Example')
ax.legend(loc='upper left')
ax.set_ylim(-0.1, 1.15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Alternate Implementations (Solution)


In [ ]:
# A. Pure Python
def sigmoid_pure(z):
    if isinstance(z, (int, float)):
        return 1 / (1 + np.e ** (-z))   # still use math but ok
    return [1 / (1 + np.e ** (-zi)) for zi in z]

# B. scipy
def sigmoid_scipy(z):
    return expit(z)

# C. Numerically stable NumPy
def sigmoid_stable(z):
    z = np.asarray(z, dtype=float)
    # clip to avoid overflow / underflow
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

z_test = np.array([-10., -1, 0, 1, 10])
print("Original :", sigmoid(z_test))
print("Pure     :", np.array(sigmoid_pure(z_test)))
print("Scipy    :", sigmoid_scipy(z_test))
print("Stable   :", sigmoid_stable(z_test))
print("Max abs diff (orig vs stable):", np.max(np.abs(sigmoid(z_test) - sigmoid_stable(z_test))))


## 6. More Practice — Solutions


In [ ]:
# 6.1 Threshold exploration
for tau in [0.3, 0.5, 0.7]:
    preds_t = (probs >= tau).astype(int)
    print(f"τ={tau}: preds={preds_t}, acc={(preds_t==y_train).mean():.2f}")

print("""
Interpretation:
- Lower τ (0.3) → more positive predictions → higher sensitivity, more false positives.
- Higher τ (0.7) → fewer positives → higher specificity, risk of missing true positives.
In medical screening you often prefer lower τ (catch more disease cases).
In spam filtering you may prefer higher τ (avoid blocking legitimate mail).
""")

# 6.2 Logistic loss
def logistic_loss(x, y, w, b, eps=1e-12):
    p = predict_proba(x, w, b)
    p = np.clip(p, eps, 1 - eps)
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return loss

print("Average logistic loss:", logistic_loss(x_train, y_train, w_in, b_in))

# 6.3 Two-feature note
print("""
2-D decision boundary:
z = w1*x1 + w2*x2 + b = 0  ⇒  x2 = (-w1/w2)*x1 - b/w2
→ a straight line in the (x1,x2) plane. Points on one side are class 1, the other class 0.
The probability is constant along lines parallel to that boundary.
""")


## 7. Simulation Section (Solution with commentary)


In [ ]:
# === PARAMETER BOX ===
np.random.seed(42)
n               = 150
w_true          = 1.5
b_true          = -6.0
threshold       = 0.5
label_noise_frac= 0.05
z_noise_std     = 0.3

x_sim = np.random.uniform(0, 10, n)
z_clean = w_true * x_sim + b_true
z_noisy = z_clean + np.random.normal(0, z_noise_std, n)
p_true  = sigmoid(z_noisy)
y_sim   = (np.random.rand(n) < p_true).astype(int)
flip_idx = np.random.choice(n, size=int(label_noise_frac * n), replace=False)
y_sim[flip_idx] = 1 - y_sim[flip_idx]

w_hat, b_hat = w_true, b_true   # oracle for illustration
probs = sigmoid(w_hat * x_sim + b_hat)
preds = (probs >= threshold).astype(int)
acc   = (preds == y_sim).mean()
print(f"Accuracy at τ={threshold}: {acc:.3f}")
print(f"Positive rate: {y_sim.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x_sim[y_sim==0], y_sim[y_sim==0], c='C0', alpha=0.55, label='y=0')
axes[0].scatter(x_sim[y_sim==1], y_sim[y_sim==1], c='C3', alpha=0.55, label='y=1')
x_line = np.linspace(0, 10, 200)
axes[0].plot(x_line, sigmoid(w_hat*x_line + b_hat), 'k-', lw=2, label='f(x)')
axes[0].axhline(threshold, color='orange', ls='--')
axes[0].set_title(f'Simulated data (acc={acc:.2f})')
axes[0].legend(); axes[0].set_xlabel('x'); axes[0].set_ylabel('y / p')

ths = np.linspace(0.1, 0.9, 17)
accs = [((sigmoid(w_hat*x_sim + b_hat) >= t).astype(int) == y_sim).mean() for t in ths]
axes[1].plot(ths, accs, 'o-', color='purple')
axes[1].axvline(threshold, color='orange', ls='--')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs Threshold')
plt.tight_layout()
plt.show()

print("""
Try editing the PARAMETER BOX:
- Increase label_noise_frac → accuracy drops, curve becomes less separable.
- Change threshold away from 0.5 → accuracy usually decreases on balanced data, but may improve if costs are asymmetric.
- Larger |w_true| → steeper transition, easier separation when noise is moderate.
""")


## 8. What the Model Can & Cannot Predict (expanded)

**Can:**
- Produce a smooth probability for a binary outcome under a linear-logit assumption.
- Support ranking of examples by risk / propensity.
- Yield interpretable coefficients (log-odds multipliers) when features are well-behaved.

**Cannot:**
- Output more than two classes without extension (one-vs-rest or softmax).
- Model highly non-linear relationships unless you add polynomial / interaction features or move to other architectures.
- Guarantee causal statements; logistic regression is associational by default.
- Automatically correct for selection bias, measurement error, or concept drift.

**Responsible checklist (audience-aware):**
- **Data-literate / technical audience:** report coefficient CIs, calibration slope, Brier score, AUC, and decision-curve analysis.
- **Executives / decision makers:** translate threshold choice into expected FP/FN counts and monetary or human cost; show a simple “what-if” table.
- **Nonspecialists / patients / public:** avoid probability jargon; use frequency statements (“about 3 out of 10 people with this profile …”) and emphasize that the model is one input among many.

Full discussion, top-10 applications, and constraints appear in the companion Word document.


## 9. Closing

You have now:
- Implemented and visualized the sigmoid
- Built a complete 1-D logistic regression predictor
- Explored numerical alternatives and threshold effects
- Run a small Monte-Carlo-style simulation of noise and threshold
- Reflected on appropriate and inappropriate uses of the model

Proceed to gradient descent on the logistic cost, multi-feature examples, or the reusable template for your own data.
